# Matrixsteifigkeitsmethode

Implementierung aller Schritte:

1. Elementsteifigkeitsmatrix
2. Inzidenztafel
3. Assemblierung
4. Randbedingungen und Loesung

> **Nur in Google Colab noetig:**

In [ ]:
import os, sys
if not os.path.exists("FEM"):
    !git clone --depth=1 -q https://github.com/Boscij/FEM.git
if "FEM/content" not in sys.path:
    sys.path.insert(0, "FEM/content")

In [ ]:
import numpy as np

## Modell

Koordinaten in [mm], Flaechen in [mm^2], E in [MPa], Kraefte in [N].

In [ ]:
nodal_coordinates = np.array([
    [   0.0,    0.0],   # Knoten 1
    [1000.0,    0.0],   # Knoten 2
    [1000.0, 1000.0],   # Knoten 3
    [2000.0, 1000.0],   # Knoten 4
])

elements = [
    [0, 1, "section 1"],
    [0, 2, "section 2"],
    [1, 2, "section 3"],
    [1, 3, "section 4"],
    [2, 3, "section 5"],
]

materials = {"steel": [210000.0]}

sections = {
    "section 1": [15.00, "steel"],
    "section 2": [28.28, "steel"],
    "section 3": [10.00, "steel"],
    "section 4": [56.56, "steel"],
    "section 5": [10.00, "steel"],
}

constraints = [
    [0, 0, 0.0],
    [0, 1, 0.0],
    [1, 1, 0.0],
]

loads = [
    [3, 1, -1000.0],
]

## 1) Elementsteifigkeitsmatrix

Lokale Steifigkeit und Transformation ins globale System:

    k_lok = (EA/L) * [[1,-1],[-1,1]]
    T     = [[c, s, 0, 0], [0, 0, c, s]]
    Ke    = T.T @ k_lok @ T

In [ ]:
def element_stiffness_matrix(EA, xy_e):
    dx = xy_e[1, 0] - xy_e[0, 0]
    dy = xy_e[1, 1] - xy_e[0, 1]
    L  = np.sqrt(dx**2 + dy**2)
    c, s = dx / L, dy / L
    k_lok = (EA / L) * np.array([[ 1., -1.], [-1.,  1.]])
    T = np.array([[c, s, 0., 0.], [0., 0., c, s]])
    return T.T @ k_lok @ T

## 2) Inzidenztafel (DOF-Mapping)

Konvention: u_x am Knoten i = DOF 2i, u_y = DOF 2i+1.
Fuer Element (i,j): [2i, 2i+1, 2j, 2j+1]

In [ ]:
def incidence_table(elements):
    conn = np.array([[e[0], e[1]] for e in elements], dtype=int)
    dofs = np.vstack((
        2*conn[:, 0], 2*conn[:, 0] + 1,
        2*conn[:, 1], 2*conn[:, 1] + 1,
    )).T
    return dofs

## 3) Assemblierung

Jedes Element traegt Ke an den zugehoerigen DOF-Positionen zur globalen K bei.

In [ ]:
def assemble_K(nodal_coordinates, elements, sections, materials):
    dofs = incidence_table(elements)
    ndof = int(np.max(dofs) + 1)
    K = np.zeros((ndof, ndof))
    for e, (i, j, sec_key) in enumerate(elements):
        A, mat_key = sections[sec_key]
        E = materials[mat_key][0]
        Ke = element_stiffness_matrix(E * A, nodal_coordinates[[i, j], :])
        K[np.ix_(dofs[e], dofs[e])] += Ke
    return K

## 4) Randbedingungen und Loesung

Aufteilen in freie (F) und gesperrte (U) DOFs:

    U_F = inv(K_FF) @ (F_F - K_FU @ U_U)

In [ ]:
def solve_system(K, constraints, loads):
    ndof = K.shape[0]
    fixed = np.zeros(ndof, dtype=bool)
    U_U = []
    for node, axis, val in constraints:
        fixed[2*int(node) + int(axis)] = True
        U_U.append(val)
    U_U = np.array(U_U, dtype=float)
    free = ~fixed
    F = np.zeros(ndof)
    for node, axis, val in loads:
        F[2*int(node) + int(axis)] = val
    U_F = np.linalg.solve(K[np.ix_(free, free)], F[free] - K[np.ix_(free, fixed)] @ U_U)
    U = np.zeros(ndof)
    U[fixed] = U_U
    U[free]  = U_F
    F[fixed] = K[np.ix_(fixed, free)] @ U_F + K[np.ix_(fixed, fixed)] @ U_U
    return U, F, fixed

## Ergebnisse

In [ ]:
K = assemble_K(nodal_coordinates, elements, sections, materials)
U, F, fixed = solve_system(K, constraints, loads)

np.set_printoptions(precision=6, suppress=True)
print("Verschiebungen U [mm]:")
for k, u in enumerate(U):
    mark = "  <- gesperrt" if fixed[k] else ""
    print(f"  U_{k+1} = {u:+.6e} mm{mark}")
print("
Kraefte F [N] (inkl. Reaktionen):")
for k, f in enumerate(F):
    mark = "  <- Reaktion" if fixed[k] else ""
    print(f"  F_{k+1} = {f:+.4f} N{mark}")